# Модуль А: Базовая кибериммунная архитектура
## Задание соревнований «КиберИммунный Светофор»

---

Вы работаете над созданием **кибериммунной системы управления светофором**.

В этом модуле вам предстоит:
1. Изучить архитектуру системы
2. Сформулировать цели и предположения безопасности по ГОСТ Р 72118-2025
3. Реализовать базовые компоненты системы
4. Запустить базовое тестирование



---

### Что такое кибериммунность?

**Кибериммунность** — это подход к разработке ПО, при котором система изначально проектируется так, чтобы быть устойчивой к атакам. Ключевое отличие от традиционной безопасности: не «починить после взлома», а «сделать взлом невозможным».

Основные принципы:
- **Минимизация доверенной базы (TCB)** — доверять только тому, что необходимо
- **Изоляция компонентов** — компоненты общаются только через защищённые каналы
- **Политики безопасности** — все действия проверяются монитором

## Описание системы

Система управляет светофором с 4 сигналами: `car_red`, `car_yellow`, `car_green`, `ped_green`.

Определите допустимые состояния светофора, исключив физически опасные комбинации сигналов.

### Компоненты системы

| Компонент | Тип | Роль |
|-----------|-----|------|
| `Monitor` | **Доверенный (TCB)** | Проверяет все события по политикам безопасности |
| `ControlSystem` | Недоверенный | Генерирует команды управления светофором |
| `LightsGPIO` | Исполнительный | Управляет физическими светодиодами |
| `PedestrianButton` | Недоверенный | Обрабатывает нажатие кнопки пешехода |
| `Event` | Структура данных | Сообщение между компонентами |

### Задание А.4 — Архитектурная диаграмма

Нарисуйте архитектурную диаграмму системы, показывающую:
- Все компоненты и их роли (доверенные / недоверенные)
- Потоки данных между компонентами
- Роль монитора безопасности в архитектуре

Вставьте диаграмму ниже (ASCII-art, mermaid или изображение):

```
Недоверенная зона       |  Доверенная зона
                        |
PedestrianButton        |   LightsGPIO
(недоверенный ввод)     | (исполнитель)
        |               |       ^
      Event             |       |
        |               |   Event (прошедший политики)
        v               |       |
   ControlSystem------------->Monitor
(логика переключения)   | (политики безопасности)
```

---
## Задание А.1: Цели и предположения безопасности

Согласно **ГОСТ Р 72118-2025**, кибериммунная система должна иметь явно сформулированные:
- **Цели безопасности (ЦБ)** — что система должна обеспечивать
- **Предположения безопасности (ПБ)** — при каких условиях это выполняется

### Ваша задача:
Заполните словари `SECURITY_GOALS` и `SECURITY_ASSUMPTIONS`, заменив пустые строки на реальные описания.

**Требование:** цели безопасности должны быть сформулированы **в формате ГОСТ** (субъект → действие → объект).

Ознакомьтесь с ГОСТ Р 72118-2025 для понимания требований к формулировкам ЦБ и ПБ.

In [ ]:
# Задание А.1: Заполните цели и предположения безопасности

SECURITY_GOALS = {
    "ЦБ-1": "Система управления светофором должна обеспечивать передачу на LightsGPIO только тех команд, которые успешно прошли верификацию в Monitor.",  # Сформулируйте цель безопасности
    "ЦБ-2": "Монитор безопасности (Monitor) должен блокировать любые команды управления, исходящие от ControlSystem, которые не соответствуют утвержденной политике безопасных состояний светофорного объекта.",  # Сформулируйте цель безопасности
    "ЦБ-3": "Система управления светофором должна предотвращать подачу на LightsGPIO команд, приводящих к конфликтному включению пешеходного зеленого (ped_green) одновременно с зеленым (car_green) или желтым (car_yellow) сигналами для автомобилей.",  # Сформулируйте цель безопасности
}

SECURITY_ASSUMPTIONS = {
    "ПБ-1": "Физический доступ к линиям управления LightsGPIO и аппаратной части Monitor ограничен и защищен от прямого вмешательства.",  # Сформулируйте предположение безопасности
    "ПБ-2": "Предполагается, что ControlSystem не имеет возможности прямого доступа к LightsGPIO в обход Monitor.",  # Сформулируйте предположение безопасности
}

# Вывод для проверки
print("Цели безопасности:")
for key, value in SECURITY_GOALS.items():
    print(f"  {key}: {value}")

print("\nПредположения безопасности:")
for key, value in SECURITY_ASSUMPTIONS.items():
    print(f"  {key}: {value}")

In [ ]:
# ===== ТЕСТ А.1 — автоматическая проверка =====

def test_security_goals():
    assert len(SECURITY_GOALS) >= 3, "Должно быть не менее 3 целей безопасности"
    for key, value in SECURITY_GOALS.items():
        assert "TODO" not in value, f"{key}: замените TODO на реальное описание"
        assert len(value) > 20, f"{key}: описание слишком короткое (минимум 20 символов)"
    print("✓ Тест А.1 пройден: цели безопасности определены корректно")


def test_security_assumptions():
    assert len(SECURITY_ASSUMPTIONS) >= 2, "Должно быть не менее 2 предположений безопасности"
    for key, value in SECURITY_ASSUMPTIONS.items():
        assert "TODO" not in value, f"{key}: замените TODO на реальное описание"
        assert len(value) > 15, f"{key}: описание слишком короткое"
    print("✓ Тест А.1б пройден: предположения безопасности определены корректно")


test_security_goals()
test_security_assumptions()
print("\n[А.1] Все тесты пройдены")

---
## Задание А.2: Класс Event и допустимые состояния

Все коммуникации в кибериммунной системе происходят только через **события (Events)**.

### Ваша задача:
1. Реализуйте класс `Event` в соответствии с архитектурой
2. Определите список допустимых состояний `ALLOWED_STATES`

**Важно:** список `ALLOWED_STATES` является «белым списком» (whitelist).
Монитор блокирует любые состояния, которых **нет** в этом списке.

In [4]:
import time
from datetime import datetime
from queue import Queue
from threading import Thread


# ─────────────────────────────────────────────────────
# Задание А.2: Реализуйте класс Event
# ─────────────────────────────────────────────────────
class Event:
    """Событие в системе. Реализуйте в соответствии с архитектурой."""
    def __init__(self, source, destination, operation: str, params: dict):
        self.source = source           
        self.destination = destination 
        self.operation = operation    
        self.params = params or {}
        self.timestamp = datetime.now()    

    def __repr__(self):
        return f"[{self.source} -> {self.destination}]: {self.operation}({self.params})"


# ─────────────────────────────────────────────────────
# Задание А.2б: Определите допустимые состояния светофора
# ─────────────────────────────────────────────────────
ALLOWED_STATES = [
    # Определите допустимые конфигурации светофора
    # Формат: (car_red, car_yellow, car_green, ped_green)
    (True, False, False, False),
    (False, True, False, False),
    (False, False, True, False),
    (True, False, False, True),
    (False, False, False, False)
]

print(f"Определено {len(ALLOWED_STATES)} допустимых состояний")

Определено 5 допустимых состояний


In [5]:
# ===== ТЕСТ А.2 =====

def test_event_class():
    # Проверка создания события
    e = Event(source="src", destination="dst", operation="test_op", params={"key": "val"})

    assert hasattr(e, 'source'), "Event должен иметь поле 'source'"
    assert hasattr(e, 'destination'), "Event должен иметь поле 'destination'"
    assert hasattr(e, 'operation'), "Event должен иметь поле 'operation'"
    assert hasattr(e, 'params'), "Event должен иметь поле 'params'"
    assert hasattr(e, 'timestamp'), "Event должен иметь поле 'timestamp'"

    assert e.source == "src", "source должен сохраняться"
    assert e.destination == "dst", "destination должен сохраняться"
    assert e.operation == "test_op", "operation должен сохраняться"
    assert e.params == {"key": "val"}, "params должен сохраняться"
    assert isinstance(e.timestamp, datetime), "timestamp должен быть типа datetime"

    repr_str = repr(e)
    assert "TODO" not in repr_str, "__repr__ должен возвращать реальное описание, не TODO"

    print("✓ Тест А.2: класс Event реализован корректно")
    print(f"  Пример события: {repr(e)}")


def test_allowed_states():
    assert len(ALLOWED_STATES) >= 4, "Должно быть не менее 4 допустимых состояний"

    # Проверка формата
    for state in ALLOWED_STATES:
        assert len(state) == 4, f"Каждое состояние — кортеж из 4 элементов: {state}"
        assert all(isinstance(v, bool) for v in state), f"Все значения должны быть bool: {state}"

    # Проверка безопасности: нет запрещённых состояний
    for state in ALLOWED_STATES:
        car_red, car_yellow, car_green, ped_green = state
        assert not (car_green and ped_green), \
            f"ОПАСНОЕ состояние: car_green=True и ped_green=True одновременно! {state}"

    # Проверка: состояния не дублируются (А.2.4)
    assert len(ALLOWED_STATES) == len(set(ALLOWED_STATES)), "FAIL: ALLOWED_STATES содержит дубликаты"

    # Обязательные состояния
    assert (True, False, False, False) in ALLOWED_STATES, \
        "Красный авто (True, False, False, False) должен быть в ALLOWED_STATES"
    assert (False, False, True, False) in ALLOWED_STATES, \
        "Зелёный авто (False, False, True, False) должен быть в ALLOWED_STATES"

    print(f"✓ Тест А.2б: ALLOWED_STATES содержит {len(ALLOWED_STATES)} безопасных состояний")


test_event_class()
test_allowed_states()
print("\n[А.2] Все тесты пройдены")

✓ Тест А.2: класс Event реализован корректно
  Пример события: [src -> dst]: test_op({'key': 'val'})
✓ Тест А.2б: ALLOWED_STATES содержит 5 безопасных состояний

[А.2] Все тесты пройдены


---
## Задание А.3: Реализация компонентов системы

Теперь реализуйте три главных компонента системы:

1. **`LightsGPIO`** — исполнительный компонент, управляет светодиодами
2. **`Monitor`** — доверенная компонента (TCB), проверяет события по политикам
3. **`ControlSystem`** — недоверенный компонент, генерирует команды

Определите необходимые методы и поля каждого компонента,
опираясь на описание архитектуры и автоматические тесты.

In [12]:
# ─────────────────────────────────────────────────────
# Задание А.3: Реализуйте компоненты системы
# ─────────────────────────────────────────────────────

class LightsGPIO:
    """Исполнительный компонент — управляет физическими сигналами светофора."""
    def __init__(self):
        self.current_state = (False, False, False, False)
        self.events_queue = Queue()
        self.running = True
    
    def run(self):
        while self.running:
            try:
                event = self.events_queue.get(timeout=0.1)
                if event and event.operation == 'set_state':
                    self.current_state = event.params['state']
            except:
                continue

    def stop(self):
        self.running = False


class Monitor:
    """Монитор безопасности — доверенная компонента (TCB).
    Обеспечивает проверку всех команд перед их исполнением."""
    def __init__(self, lights):
        self.lights = lights
        self.policies = []
        self.violations_log = []
        self.events_queue = Queue()
        self.running = True

    def add_policy(self, policy):
        self.policies.append(policy)

    def check_event(self, event):
        for policy in self.policies:
            if not policy(event):
                self._log_violation(event, policy)
                return False
        return True
    
    def _log_violation(self, event, policy):
        violation = {
            'timestamp': datetime.now(),
            'event': event,
            'policy': policy.__name__,
            'reason': f'Политика {policy.__name__} отклонена'
        }
        self.violations_log.append(violation)

    def run(self):
        while self.running:
            try:
                event = self.events_queue.get(timeout=0.5)
                if self.check_event(event):
                    self.lights.events_queue.put(event)
            except:
                continue
    
    def stop(self):
        self.running = False


class ControlSystem:
    """Система управления — недоверенный компонент.
    Взаимодействует с LightsGPIO ТОЛЬКО через Monitor."""
    def __init__(self, monitor):
        self.monitor = monitor
        
    
    def request_state_change(self, state):
        event = Event(
            source=self,
            destination=self.monitor,
            operation='set_state',
            params={'state': state}
        )

        self.monitor.events_queue.put(event)

print("Классы определены. Запустите тест ниже.")

Классы определены. Запустите тест ниже.


In [9]:
# Политика whitelist — используется ниже в тестах
def whitelist_policy(event):
    """Политика безопасности: проверяет допустимость запрошенного состояния."""
    if event.operation != 'set_state':
        return True
    return event.params['state'] in ALLOWED_STATES


print("Политика определена.")

Политика определена.


In [13]:
# ===== ТЕСТ А.3 =====

def test_monitor_allows_valid_state():
    """Тест: монитор пропускает допустимое состояние."""
    lights = LightsGPIO()
    monitor = Monitor(lights)
    monitor.add_policy(whitelist_policy)

    # Запуск в потоках
    t_lights = Thread(target=lights.run, daemon=True)
    t_monitor = Thread(target=monitor.run, daemon=True)
    t_lights.start()
    t_monitor.start()

    ctrl = ControlSystem(monitor)

    # Отправляем допустимое состояние
    ctrl.request_state_change((False, False, True, False))  # зелёный авто
    time.sleep(0.3)

    assert lights.current_state == (False, False, True, False), \
        f"Монитор должен пропустить допустимое состояние. Текущее: {lights.current_state}"
    assert len(monitor.violations_log) == 0, "Нарушений быть не должно"

    print("✓ Тест А.3а: монитор корректно пропускает допустимое состояние")


def test_monitor_blocks_invalid_state():
    """Тест: монитор блокирует запрещённое состояние."""
    lights = LightsGPIO()
    monitor = Monitor(lights)
    monitor.add_policy(whitelist_policy)

    t_lights = Thread(target=lights.run, daemon=True)
    t_monitor = Thread(target=monitor.run, daemon=True)
    t_lights.start()
    t_monitor.start()

    ctrl = ControlSystem(monitor)
    initial_state = lights.current_state

    # Отправляем ЗАПРЕЩЁННОЕ состояние: зелёный авто + зелёный пешеход
    ctrl.request_state_change((False, False, True, True))
    time.sleep(0.3)

    assert lights.current_state == initial_state, \
        f"Монитор ДОЛЖЕН заблокировать опасное состояние! Текущее: {lights.current_state}"
    assert len(monitor.violations_log) >= 1, \
        "Нарушение должно быть зафиксировано в журнале"

    print("✓ Тест А.3б: монитор корректно блокирует запрещённое состояние")
    print(f"  Нарушений зафиксировано: {len(monitor.violations_log)}")


test_monitor_allows_valid_state()
test_monitor_blocks_invalid_state()
print("\n[А.3] Все тесты пройдены")

✓ Тест А.3а: монитор корректно пропускает допустимое состояние
✓ Тест А.3б: монитор корректно блокирует запрещённое состояние
  Нарушений зафиксировано: 1

[А.3] Все тесты пройдены


: 

---
## Итоги Модуля А

Если все тесты пройдены, модуль завершён!

| Задание | Тест                                          |
|-------------|-----------------------------------------------|
| А.1 Цели безопасности | `test_security_goals()`                       |
| А.2 Event + ALLOWED_STATES | `test_event_class()`, `test_allowed_states()` |
| А.3 Monitor + ControlSystem + LightsGPIO | `test_monitor_*`                              |